# OMRモデルの推論

In [16]:
import yaml
import torch
import sys
import os

sys.path.append("..")
from yolov3.models.yolo import (
    BaseModel,
)  # notebookの場合パスが通らないのでこのようにする
from src.domain.model import OMRModel
from src.domain.dataloader import CustomDataset, custom_collate_fn
from src.domain.loss import CustomLoss
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim

from torchvision import transforms
from PIL import Image
from src.utils import non_max_suppression
import cv2

In [23]:
model_dir = "../models/"
config_path = model_dir + "config/omr_yolov5s.yaml"  #'yolov3/models/yolov5s.yaml'
model_path = model_dir + "pre_trained/yolov5s.pt"

data_dir = "../data/"
hyp_path = data_dir + "hyps/hyp.scratch-low.yaml"
img_dir = data_dir + "dense/images/"
annotation_dir = data_dir + "dense/labels_mapping/"

EXP_NAME = "250508_complete_weightedCIoU_epoch10"
fine_tuned_path = model_dir + f"fine_tuned/{EXP_NAME}/epoch2/omr_yolov5s.pth"

ratio_wh = 1
RESIZE_SIZE_W = 1280
RESIZE_SIZE_H = int(ratio_wh * RESIZE_SIZE_W)
print(RESIZE_SIZE_H, RESIZE_SIZE_W)

1280 1280


In [24]:
model = OMRModel(config_path)
model.load_state_dict(torch.load(fine_tuned_path))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 推論モード
model.eval()

image_path = data_dir + "dense/images/1.png"
im0 = cv2.imread(image_path)
image = Image.open(image_path).convert("RGB")
transform = transforms.Compose(
    [
        transforms.Resize((RESIZE_SIZE_W, RESIZE_SIZE_H)),
        transforms.ToTensor(),
    ]
)
image = transform(image).unsqueeze(0)
# 推論
with torch.no_grad():  # メモリ効率を向上させるためにtorch.no_gradを使用
    input = image.to(device)
    pred = model(input)

# 推論結果を表示または処理するコードを追加
print(pred)


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  yolov3.models.common.Conv               [3, 32, 6, 2, 2]              
  1                -1  1     18560  yolov3.models.common.Conv               [32, 64, 3, 2]                
  2                -1  1     18816  yolov3.models.common.C3                 [64, 64, 1]                   
  3                -1  1     73984  yolov3.models.common.Conv               [64, 128, 3, 2]               
  4                -1  2    115712  yolov3.models.common.C3                 [128, 128, 2]                 
  5                -1  1    295424  yolov3.models.common.Conv               [128, 256, 3, 2]              
  6                -1  3    625152  yolov3.models.common.C3                 [256, 256, 3]                 
  7                -1  1   1180672  yolov3.models.common.Conv               [256, 512, 3, 2]              
  8                -1  1   1182720  

omr_YOLOv3s summary: 214 layers, 7456543 parameters, 7456543 gradients, 17.3 GFLOPs



(tensor([[[3.51221e+00, 3.45919e+00, 1.16836e+01,  ..., 1.90312e-01, 1.72586e-01, 1.62803e-01],
         [1.00262e+01, 3.07860e+00, 1.38456e+01,  ..., 2.19702e-01, 1.97297e-01, 1.88321e-01],
         [1.94849e+01, 2.70659e+00, 1.37879e+01,  ..., 1.98449e-01, 1.77349e-01, 1.73199e-01],
         ...,
         [1.19953e+03, 1.28176e+03, 1.74164e+01,  ..., 2.72135e-03, 1.49135e-06, 7.76981e-04],
         [1.21739e+03, 1.25870e+03, 1.20689e+01,  ..., 4.67424e-05, 1.12239e-06, 1.93057e-04],
         [1.27106e+03, 1.29332e+03, 2.16253e+01,  ..., 2.38669e-03, 1.50048e-06, 2.06003e-04]]], device='cuda:0'), [tensor([[[[[-1.22100e-01, -1.35408e-01,  6.01351e-01,  ..., -1.44798e+00, -1.56741e+00, -1.63752e+00],
           [-5.03847e-01, -2.31377e-01,  8.62570e-01,  ..., -1.26741e+00, -1.40327e+00, -1.46095e+00],
           [-1.28955e-01, -3.26214e-01,  8.55550e-01,  ..., -1.39602e+00, -1.53441e+00, -1.56312e+00],
           ...,
           [-2.43640e-01, -3.10270e-01,  8.10980e-01,  ..., -1.38329e

In [25]:
print(isinstance(pred, (list, tuple)))
print("len(pred)", len(pred))
print("pred[0]のshape", pred[0].shape)  # nc+npitch+xyhw+conf
print("len(pred[1])", len(pred[1]))
print(pred[1][0].shape)
print(pred[1][1].shape)
print(pred[1][2].shape)
print(pred[0][0][0])

print("conf max", pred[0][..., 4].max())
print("x max", pred[0][..., 0].min())
print(pred[0][pred[0][..., 4] > 0.2][:, 4])

True
len(pred) 2
pred[0]のshape torch.Size([1, 100800, 167])
len(pred[1]) 3
torch.Size([1, 3, 160, 160, 167])
torch.Size([1, 3, 80, 80, 167])
torch.Size([1, 3, 40, 40, 167])
tensor([3.51221e+00, 3.45919e+00, 1.16836e+01, 7.89519e+00, 1.68507e-01, 3.61969e-03, 4.38332e-03, 5.85462e-03, 4.52944e-03, 3.37999e-03, 3.45561e-03, 5.35949e-02, 1.29739e-01, 3.30826e-02, 1.34056e-01, 3.59691e-03, 3.87211e-03, 6.44549e-03, 3.76171e-03, 3.69322e-03, 3.74602e-03, 3.52385e-03, 7.48339e-03, 4.36683e-03,
        4.51310e-03, 4.37800e-03, 4.42038e-03, 4.15868e-03, 3.98907e-03, 5.47344e-03, 4.53628e-01, 4.60035e-03, 4.51348e-01, 3.97714e-03, 2.77005e-01, 5.39583e-03, 2.70499e-01, 4.25994e-03, 2.16294e-01, 3.56677e-03, 2.15098e-01, 5.50864e-03, 6.44889e-02, 3.96041e-03, 6.07204e-02, 5.69338e-03, 4.33876e-03, 3.70228e-03,
        4.11261e-03, 4.38401e-03, 5.72702e-03, 6.55050e-03, 3.33642e-03, 2.70717e-03, 4.45546e-03, 4.16589e-03, 4.09434e-03, 3.67448e-03, 3.49725e-03, 4.45141e-03, 6.11506e-03, 3.52310e-0

In [26]:
print("w", im0.shape[1])
print("h", im0.shape[0])

w 1960
h 2772


In [27]:
output = non_max_suppression(
    pred,
    conf_thres=0.10,
    w=im0.shape[1] / RESIZE_SIZE_W,
    h=im0.shape[0] / RESIZE_SIZE_H,
)
print(output[0].shape)
print(output[0])
print(output[0].cpu().numpy())

x tensor([[3.51221e+00, 3.45919e+00, 1.16836e+01,  ..., 1.90312e-01, 1.72586e-01, 1.62803e-01],
        [1.00262e+01, 3.07860e+00, 1.38456e+01,  ..., 2.19702e-01, 1.97297e-01, 1.88321e-01],
        [1.94849e+01, 2.70659e+00, 1.37879e+01,  ..., 1.98449e-01, 1.77349e-01, 1.73199e-01],
        ...,
        [1.73311e+02, 1.21847e+03, 2.82602e+01,  ..., 1.50477e-06, 1.19591e-06, 2.95520e-05],
        [8.45703e+01, 1.23876e+03, 2.87746e+01,  ..., 2.42243e-06, 1.23524e-06, 1.54449e-05],
        [1.16585e+02, 1.23566e+03, 1.71006e+01,  ..., 1.38758e-06, 1.14929e-06, 2.07598e-05]], device='cuda:0')
x tensor([[3.51221e+00, 3.45919e+00, 1.16836e+01,  ..., 3.20690e-02, 2.90820e-02, 2.74334e-02],
        [1.00262e+01, 3.07860e+00, 1.38456e+01,  ..., 2.46576e-02, 2.21431e-02, 2.11357e-02],
        [1.94849e+01, 2.70659e+00, 1.37879e+01,  ..., 2.51280e-02, 2.24563e-02, 2.19309e-02],
        ...,
        [1.73311e+02, 1.21847e+03, 2.82602e+01,  ..., 3.17802e-07, 2.52571e-07, 6.24126e-06],
        [8.4

In [28]:
from ultralytics.utils.plotting import Annotator, colors
from yolov3.utils.general import (
    cv2,
    scale_boxes,
)

save_crop = False  # save cropped prediction boxes
line_thickness = 3  # bounding box thickness (pixels)
save_txt = False  # save results to *.txt
view_img = True  # show results
nosave = False  # do not save images/videos
hide_labels = False  # hide labels
hide_conf = False  # hide confidences

names = model.names
# im0 = im0.to(device)
im = image.to(device)
output_path = "../data/output/dense/detection/"
os.makedirs(output_path, exist_ok=True)

print("読み込み完了")
# Process predictions
for i, det in enumerate(output):  # per image
    print("処理開始")
    # det = det.cpu()
    gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]  # normalization gain whwh
    imc = im0.copy() if save_crop else im0  # for save_crop
    annotator = Annotator(im0, line_width=line_thickness, example=str(names))
    if len(det):
        # Rescale boxes from img_size to im0 size
        # det[:, :4] = scale_boxes(im.shape[2:], det[:, :4], im0.shape).round()
        print("det", det)

        # Print results
        for c in det[:, 5].unique():
            n = (det[:, 5] == c).sum()  # detections per class
            # s += f"{n} {names[int(c)]}{'s' * (n > 1)}, "  # add to string

        # Write results
        for *xyxy, conf, cls in reversed(det):
            # print("annotator")
            # if save_txt:  # Write to file
            #     xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(-1).tolist()  # normalized xywh
            #     line = (cls, *xywh, conf) if save_conf else (cls, *xywh)  # label format
            #     with open(f"{txt_path}.txt", "a") as f:
            #         f.write(("%g " * len(line)).rstrip() % line + "\n")

            c = int(cls.cpu())  # integer class
            label = (
                None
                if hide_labels
                else (names[c] if hide_conf else f"{names[c]} {conf:.2f}")
            )
            annotator.box_label(xyxy, label, color=colors(c, True))

    result_image = annotator.result()
    # cv2.imshow("result", result_image)
    cv2.imwrite(output_path + f"1_{EXP_NAME}.png", result_image)

読み込み完了
処理開始
det tensor([[1.07987e+02, 1.02750e+02, 1.49132e+02, 2.17324e+02, 2.65618e-01, 6.00000e+00],
        [2.59773e+02, 7.34447e+02, 2.80674e+02, 7.51126e+02, 2.09409e-01, 2.50000e+01],
        [1.24471e+03, 9.40408e+02, 1.26436e+03, 9.56765e+02, 2.02196e-01, 2.50000e+01],
        ...,
        [1.64276e+02, 2.51928e+03, 1.80808e+02, 2.57319e+03, 1.17257e-01, 6.80000e+01],
        [1.64276e+02, 2.44998e+03, 1.80808e+02, 2.50389e+03, 1.17256e-01, 6.80000e+01],
        [1.64276e+02, 2.31138e+03, 1.80808e+02, 2.36529e+03, 1.17256e-01, 6.80000e+01]], device='cuda:0')
